In [38]:
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.preprocessing import OneHotEncoder
import os
import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score
from keras.callbacks import EarlyStopping

# Definir as variáveis de caminho
DATADIR = 'C:/Users/Usuário/Downloads/machineLearning/dataset/'
TABLEDIR = DATADIR+'data.csv'
IMAGEDIR = DATADIR+'images'

# Ler os valores tabulares
df = pd.read_csv(TABLEDIR)
# Verificar o número total de entradas
print("O número total de entradas é: ", df.shape[0], "\n")
# Verificar celulas vazias na tabela
print("O número de células vazias por coluna é: \n", df.isnull().sum())

O número total de entradas é:  33126 

O número de células vazias por coluna é: 
 image_name                         0
patient_id                         0
sex                               65
age_approx                        68
anatom_site_general_challenge    527
diagnosis                          0
benign_malignant                   0
target                             0
dtype: int64


In [39]:
# Como podemos ver, algumas das células não possuem valor. A coluna com a maior concentração de células vazias é anatom_site_general_challenge
# A quantidade de células vazias corresponde a 1.59% das entradas.
uniqueAnatom = df["anatom_site_general_challenge"].value_counts(dropna=False)
print(uniqueAnatom, "\n")

# Para não perdermos amostras, as células vazias categóricas (anatom e sex) serão preenchidas como 'unknown', enquanto as células vazias contínuas (age) serão preenchidas com a média
mean_age = df["age_approx"].mean()
df["age_approx"] = df["age_approx"].fillna(mean_age)
df = (df.fillna("unknown"))
print("O número de células vazias por coluna é: \n", df.isnull().sum())

anatom_site_general_challenge
torso              16845
lower extremity     8417
upper extremity     4983
head/neck           1855
NaN                  527
palms/soles          375
oral/genital         124
Name: count, dtype: int64 

O número de células vazias por coluna é: 
 image_name                       0
patient_id                       0
sex                              0
age_approx                       0
anatom_site_general_challenge    0
diagnosis                        0
benign_malignant                 0
target                           0
dtype: int64


In [40]:
# Podemos notar que algumas colunas possuem valores de texto, então iremos mapear 
# nota: Originalmente estava usando o LabelEncoder mas depois de pesquisar mais, descobri que o OneHotEncoder é melhor nesse caso.
#       As colunas 'diagnosis' e 'benign_malignant' não vão ser codificadas, elas serão removidas posteriosmente pois elas podem causar vazamento de dados

encoder = OneHotEncoder(sparse_output=False)

encoded = encoder.fit_transform(df[['anatom_site_general_challenge']])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(['anatom_site_general_challenge']), index=df.index)
df = pd.concat([df.drop(columns=['anatom_site_general_challenge']), encoded_df], axis=1)

encoded = encoder.fit_transform(df[['sex']])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(['sex']), index=df.index)
df = pd.concat([df.drop(columns=['sex']), encoded_df], axis=1)

In [41]:
# Agora vamos iniciar o pre-processamento das imagens
# Primeiro vamos criar uma nova coluna "caminho_imagem" para definir onde a imagem atrelada aquela entrada se encontra

pd.set_option('display.max_colwidth', None)

# Ajustando o caminho
df["caminho_imagem"] = df["image_name"].apply(lambda x: os.path.join(IMAGEDIR, f"{x}.jpg"))
df["caminho_imagem"] = df["caminho_imagem"].str.replace("\\", "/", regex=False)
print(df[["image_name", "caminho_imagem"]].head())


# Agora vamos verificar se todas as entradas possuem uma imagem relacionada
df["existe"] = df["caminho_imagem"].apply(os.path.exists)
print("\n", df["existe"].value_counts())
# Se uma entrada não possuir imagem, ela será removida
df = df[df["existe"]].copy()
print(df.shape)


     image_name  \
0  ISIC_2637011   
1  ISIC_0015719   
2  ISIC_0052212   
3  ISIC_0068279   
4  ISIC_0074268   

                                                               caminho_imagem  
0  C:/Users/Usuário/Downloads/machineLearning/dataset/images/ISIC_2637011.jpg  
1  C:/Users/Usuário/Downloads/machineLearning/dataset/images/ISIC_0015719.jpg  
2  C:/Users/Usuário/Downloads/machineLearning/dataset/images/ISIC_0052212.jpg  
3  C:/Users/Usuário/Downloads/machineLearning/dataset/images/ISIC_0068279.jpg  
4  C:/Users/Usuário/Downloads/machineLearning/dataset/images/ISIC_0074268.jpg  

 existe
True    33126
Name: count, dtype: int64
(33126, 18)


In [42]:
# Agora vamos definir a CNN (Convolutional Neural Network) para fazer o processar as imagens

# As imagens usadas terão dimensões 128x128 e 3 canais (RGB)
image_input = keras.Input(shape=(128, 128, 3))

# A primeira camada tem 32 filtros de tamano 3 (3x3). Ela vai detectar bordas, cantos, textura, etc
x = keras.layers.Conv2D(32, 3, activation="relu")(image_input)
# A segunda camada vai reduzir o tamanho para reduzir ruído e custo computacional
x = keras.layers.MaxPooling2D()(x)
# A terceira camada tem 64 filtros de tamano 3 (3x3). A gora vai analisar formas e padrões
x = keras.layers.Conv2D(64, 3, activation="relu")(x)
# A quarta camada faz a redução novamente
x = keras.layers.MaxPooling2D()(x)
# A quinta camada vai transformar todos os dados em um vetor
x = keras.layers.Flatten()(x)

# Por fim se cria a representação final da imagem
image_features = keras.layers.Dense(128, activation="relu")(x)

In [43]:
# Definindo agora a MLP (Multilayer Perceptron)
# Podemos ver que cada amostra será composta por um vetor de 11 variáveis
tabular_input = keras.Input(shape=(11,))

# A primeira camada vai tarnsformar a entrada em 128 "representações internas"
y = keras.layers.Dense(128, activation="relu")(tabular_input)
# A segunda camada vai eliminar redundâncias e manter padrões importantes
y = keras.layers.Dense(64, activation="relu")(y)
# A terceira camada vai eliminar redundâncias e manter padrões importantes
y = keras.layers.Dense(32, activation="relu")(y)

# Por fim temos o vertor compacto final
tabular_features = keras.layers.Dense(16, activation="relu")(y)


# nota: o número de filtros/featues da CNN vai aumentando pois a os dados que são extraídos da imagem 
#       vão se tornando cada vez mais abstratos. Enquanto o da MLP vão reduzindo para evitar overfitting e forçar generalização

In [44]:
# Aqui nós combinados a CNN e MLP, criando uma função multimodal
# Os dados de entrada serão a concatenação do vertor vindo da CNN e do MLP
combined = keras.layers.concatenate([image_features, tabular_features])

z = keras.layers.Dense(128, activation="relu")(combined)
z = keras.layers.Dense(64, activation="relu")(z)
z = keras.layers.Dense(32, activation="relu")(z)
z = keras.layers.Dropout(0.3)(z)

output = keras.layers.Dense(1, activation="sigmoid")(z)

In [45]:
# Agora iremos criar o modelo multimodal.
model = keras.Model(
    inputs=[image_input, tabular_input],
    outputs=output
)

# Aqui nós definimos a compilação do modelo (como aprender, medir erro, etc)
# O otimizadoor 'adam', vai ajustar os pesos depois que o erro é calculado
# Afunção de perda 'binary_crossentropy', é usada para calcular o erro onde a saída é um valor binário (bom para o nosso caso)
# A acurácia não afeta o treinamento, é mais usada como um 'feedback' de como está indo
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [46]:
# Aqui nós iremos abrir as imagens (garantindo que estejam em RGB) e definir o tamanho delas como 128x128
# Depois vamos converter a imagem pra um array numpy para poder ser normalizado
# Por fim aquela imagem é adicionada em um array

X_images = []
for path in df["caminho_imagem"]:
    img = Image.open(path).convert("RGB")
    img = img.resize((128, 128))
    img = np.array(img)
    img = img / 255.0

    X_images.append(img)

X_images = np.array(X_images)

In [47]:
# Aqui nós definimos os valores tabulares e o nosso objetivo (target)
# Algumas colunas da tabela não serão usadas pois não representão algo relevante ou podem levar ao vazamento de dados (como a coluna diagnosis)

X_tabular = df.drop(columns=["image_name","patient_id","caminho_imagem","existe","benign_malignant", "target", "diagnosis"])
y = df["target"]

In [48]:
# Agora, vamos separar os conjuntos de treino, validação e teste

X_train_val_tab, X_test_tab, X_train_val_img, X_test_img, y_train_val, y_test = train_test_split(
    X_tabular,
    X_images,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_train_tab, X_val_tab, X_train_img, X_val_img, y_train, y_val = train_test_split(
    X_train_val_tab,
    X_train_val_img,
    y_train_val,
    test_size=0.2,
    stratify=y_train_val,
    random_state=42
)

In [49]:
# Depois de separados, nós iremos fazer a normalização dos valores tabulares
scaler = StandardScaler()

X_train_tab = scaler.fit_transform(X_train_tab)
X_val_tab = scaler.transform(X_val_tab)
X_test_tab = scaler.transform(X_test_tab)

In [50]:
# Verificar se os dados estão balanceados
print(y.value_counts())
print(y.value_counts(normalize=True) * 100)

target
0    32542
1      584
Name: count, dtype: int64
target
0    98.237034
1     1.762966
Name: proportion, dtype: float64


In [51]:
# Como vimos acima, os dados estão desbalanceados
# Inicialmente pensei em usar o SMOTE, mas depois de pesquisardescobri que o class_weight é mais recomendado em arquiteturas multimodais
# O compute_class_weight vai calcular os pesos das classes (0/1 ; benigno/maligno)
# Isso só deve ser usado nos dados de treino para evitar vazamento de dados

classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weight = dict(zip(classes, weights))
print(class_weight)

{np.int64(0): np.float64(0.508979160664554), np.int64(1): np.float64(28.342245989304814)}


In [52]:
# Setando um early stop como boa prática (interromper quando o modelo parar de melhorar)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [53]:
# Aqui o modelo vai passar pelo treinamento. REcebendo os valores de imagem e tabular como entrada
# A epoch define quantas vezes o modelo vai percorrer o conjunto de dados
# O batch_size vai definir o número de amostras que serão vistas antes de se atualizar os pessos por epoch
# class_weight ajuda no desbalanceamento como falado antes 

history = model.fit(
    [X_train_img, X_train_tab],
    y_train,
    epochs=50,
    batch_size=32,
    class_weight=class_weight,
    callbacks=[early_stop],
    validation_data=([X_val_img, X_val_tab], y_val)
)

Epoch 1/50
663/663 ━━━━━━━━━━━━━━━━━━━━ 99s 139ms/step - accuracy: 0.5036 - loss: 0.7365 - val_accuracy: 0.0213 - val_loss: 0.8045
Epoch 2/50
663/663 ━━━━━━━━━━━━━━━━━━━━ 88s 133ms/step - accuracy: 0.4815 - loss: 0.6774 - val_accuracy: 0.6317 - val_loss: 0.5800
Epoch 3/50
663/663 ━━━━━━━━━━━━━━━━━━━━ 88s 133ms/step - accuracy: 0.6271 - loss: 0.6535 - val_accuracy: 0.5104 - val_loss: 0.7158
Epoch 4/50
663/663 ━━━━━━━━━━━━━━━━━━━━ 89s 134ms/step - accuracy: 0.6397 - loss: 0.6450 - val_accuracy: 0.6430 - val_loss: 0.6637
Epoch 5/50
663/663 ━━━━━━━━━━━━━━━━━━━━ 90s 136ms/step - accuracy: 0.6254 - loss: 0.6289 - val_accuracy: 0.3802 - val_loss: 0.8482
Epoch 6/50
663/663 ━━━━━━━━━━━━━━━━━━━━ 90s 135ms/step - accuracy: 0.6561 - loss: 0.6621 - val_accuracy: 0.5491 - val_loss: 0.7227
Epoch 7/50
663/663 ━━━━━━━━━━━━━━━━━━━━ 90s 135ms/step - accuracy: 0.6183 - loss: 0.6123 - val_accuracy: 0.7183 - val_loss: 0.5426
Epoch 8/50
663/663 ━━━━━━━━━━━━━━━━━━━━ 90s 136ms/step - accuracy: 0.6687 - loss: 0

In [55]:
# Aqui será feita a predição no conjunto de teste. Como output temos:
# Precision: Quantas entradas positivas o modelo previu corretamente
# Recall: Dos positivos reais, quantos o modelo conseguiu encontrar
# F1-score: média harmonica entre precision e recall
# Support: quantidade de entradas reais de cada classe
# Accuracy: porcentagem total de acertos
# Macro_avg: média simples das métricas das classes
# Weighted_avg: média das classes mas ponderando o número de exemplos de cada classe

y_pred = (model.predict([X_test_img,X_test_tab]) > 0.6).astype(int)

print(classification_report(y_test, y_pred))

# nota: O limiar de decisão padrão para classificadores binários é 0.5; mas o desbalanceamento está muito forte
#       eu avaliei outros (0.5 ; 0.6 ; 0.7 ; 0.8) e o que teve o melhor equilibrio entre precision e recall foi 0.6

208/208 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step
              precision    recall  f1-score   support

           0       0.99      0.90      0.94      6509
           1       0.07      0.44      0.12       117

    accuracy                           0.89      6626
   macro avg       0.53      0.67      0.53      6626
weighted avg       0.97      0.89      0.93      6626



In [56]:
# Aqui nós avaliamos o modelo com o conjunto de teste
model.evaluate([X_test_img, X_test_tab], y_test)

208/208 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - accuracy: 0.7671 - loss: 0.4529


[0.45289814472198486, 0.7671294808387756]

In [57]:
# Aqui nós calculamos o ROCA AUC
# Essa é uma métrica usada para avaliar classificadores binários. Medindo a capacidade do modelo de separar as duas classes.

probs = model.predict([X_test_img,X_test_tab])
auc = roc_auc_score(y_test, probs)
print("ROC AUC =", auc)

208/208 ━━━━━━━━━━━━━━━━━━━━ 6s 27ms/step
ROC AUC = 0.7608505251768426


In [58]:
# nota: O modelo não está perfeito. Mesmo aplicando tecnicas para o desbalanceamento (class_weight),
#       os resultados para o caso maligno ainda estão ruins. Mas esse foi o melhor que pude fazer.